In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import joblib
import pandas as pd
import seaborn as sns
import optuna
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
from sklearn.model_selection import learning_curve
from sklearn.metrics import make_scorer, classification_report
np.random.seed(42)

In [2]:
# Läs in träningsfilen
df = pd.read_csv(r"C:\Users\Eier\OneDrive\Dokument\DataScience\Data_science_projektkurs\archive (1)\diabetes_prediction_dataset.csv")
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df['gender'].unique())
print(df['smoking_history'].unique())
df_encoded = pd.get_dummies(df, columns=['gender', 'smoking_history'], drop_first=False)
print(df_encoded.head())
print(df_encoded.columns.tolist())
numeric_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 
                'HbA1c_level', 'blood_glucose_level', 'diabetes']

df_numeric = df_encoded[numeric_cols]
print(df_numeric.describe().T)
print(df['diabetes'].value_counts())

Y = df_encoded['diabetes']
X = df_encoded.drop(columns=['diabetes'])

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB
None
                 age  hypertension  heart_disease            bmi  \
count  100000.000000  100000.00000  100000.000000  100000.000000   
mean       41.885856       0.07485       0.039420      27.320767   
std        22.516840       0.26315       0.194593       6.636783   
min   

In [4]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

# 15% val, 15% test
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42, stratify=Y_temp
)

print(X_train.shape, Y_train.shape)
print(X_val.shape, Y_val.shape)
print(X_test.shape, Y_test.shape)


(70000, 15) (70000,)
(15000, 15) (15000,)
(15000, 15) (15000,)


In [5]:
hgbc = Pipeline([
('scaler', StandardScaler()),
    ('model', HistGradientBoostingClassifier())
])
hgbc.fit(X_train, Y_train)
score = hgbc.score(X_val, Y_val)
print('Modellen är tränad, och sparad.')
print("HistGradientBoostingClassifier:", score)
joblib.dump(hgbc, 'hgbc_trained.pkl')

Modellen är tränad, och sparad.
HistGradientBoostingClassifier: 0.9730666666666666


['hgbc_trained.pkl']

In [6]:
# Ladda den tränade modellen
hgbc_pipeline = joblib.load('hgbc_trained.pkl')

# Gör prediktioner på valideringsdata med den optimerade modellen
y_val_pred = hgbc_pipeline.predict(X_val)

# Utvärdera modellens prestanda
accuracy = accuracy_score(Y_val, y_val_pred)
precision = precision_score(Y_val, y_val_pred, average='macro')
recall = recall_score(Y_val, y_val_pred, average='macro')
f1= f1_score(Y_val, y_val_pred, average='weighted')

# Samla resultaten
results_hgbc = [{
    'Model': 'HistGradientBoostingClassifier',
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'f1-score': f1
}]

# Spara resultaten
joblib.dump(results_hgbc, 'hgbc_results.pkl')

# Konvertera listan till en DataFrame och visa resultaten
results_hgbc = pd.DataFrame(results_hgbc)
results_hgbc

,Model,Accuracy,Precision,Recall,f1-score
0,HistGradientBoostingClassifier,0.973067,0.974418,0.849395,0.971027


In [ ]:
Optimering av hyperparametrar med Optuna FÖR HistGradientBoostingClassifier


In [ ]:

from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# Convergence stopper
# -------------------------------------------------------
class ConvergenceStopper:
    def __init__(self, threshold=0.002, patience=5):
        self.threshold = threshold
        self.patience = patience
        self.last_best = None
        self.bad_count = 0

    def __call__(self, study, trial):
        current = trial.value
        best = study.best_value

        if self.last_best is None:
            self.last_best = best
            return

        improvement = best - self.last_best

        if improvement < self.threshold:
            self.bad_count += 1
        else:
            self.bad_count = 0
            self.last_best = best

        if self.bad_count >= self.patience:
            print(f"\n[OPTUNA] {self.patience} dåliga trials i rad → stoppar studien.")
            study.stop()

# -------------------------------------------------------
# HGB hyperparameter space
# -------------------------------------------------------
def hgb_param_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.05),
        "max_depth": trial.suggest_int("max_depth", 5, 8),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 32, 96),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 20, 60),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.5, 3.0),
        "max_bins": trial.suggest_int("max_bins", 128, 255),
    }

# -------------------------------------------------------
# Objective-funktion
# -------------------------------------------------------
def objective_hgb(trial):
    params = hgb_param_space(trial)

    model = HistGradientBoostingClassifier(
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        max_leaf_nodes=params["max_leaf_nodes"],
        min_samples_leaf=params["min_samples_leaf"],
        l2_regularization=params["l2_regularization"],
        max_bins=params["max_bins"],
        random_state=535
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    score = f1_score(y_val, preds)

    trial.set_user_attr("best_model", model)

    return score

# -------------------------------------------------------
# Train/val split
# -------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train, Y_train, test_size=0.2, random_state=535
)

# -------------------------------------------------------
# Kör Optuna-studien
# -------------------------------------------------------
stopper = ConvergenceStopper(threshold=0.002, patience=5)

study_hgb = optuna.create_study(direction="maximize")
study_hgb.optimize(objective_hgb, n_trials=100, callbacks=[stopper])

# -------------------------------------------------------
# Hämta bästa modellen
# -------------------------------------------------------
best_hgb = study_hgb.best_trial.user_attrs["best_model"]

print("\nBästa HGB-parametrar:")
print(study_hgb.best_params)

# -------------------------------------------------------
# Spara modellen med joblib
# -------------------------------------------------------
joblib.dump(best_hgb, "best_hgb_model.joblib")
print("\n[JOBLIB] Modellen sparad som best_hgb_model.joblib")


[I 2026-10-04 18:31:45,753] A new study created in memory with name: no-name-aeb7f7c7-ebd0-4314-aa36-cfa409352b61
[I 2026-10-04 18:31:46,166] Trial 0 finished with value: 0.8111455108359134 and parameters: {'learning_rate': 0.04390757872202261, 'max_depth': 7, 'max_leaf_nodes': 36, 'min_samples_leaf': 50, 'l2_regularization': 2.9592406367716313, 'max_bins': 146}. Best is trial 0 with value: 0.8111455108359134.
[I 2026-10-04 18:31:46,615] Trial 1 finished with value: 0.8129829984544049 and parameters: {'learning_rate': 0.034503669213693314, 'max_depth': 8, 'max_leaf_nodes': 57, 'min_samples_leaf': 42, 'l2_regularization': 2.3421783857225593, 'max_bins': 130}. Best is trial 1 with value: 0.8129829984544049.
[I 2026-10-04 18:31:46,971] Trial 2 finished with value: 0.8083116883116883 and parameters: {'learning_rate': 0.04208597760525254, 'max_depth': 5, 'max_leaf_nodes': 96, 'min_samples_leaf': 45, 'l2_regularization': 2.587210597888389, 'max_bins': 170}. Best is trial 1 with value: 0.8129


[OPTUNA] 5 dåliga trials i rad → stoppar studien.

Bästa HGB-parametrar:
{'learning_rate': 0.034503669213693314, 'max_depth': 8, 'max_leaf_nodes': 57, 'min_samples_leaf': 42, 'l2_regularization': 2.3421783857225593, 'max_bins': 130}

[JOBLIB] Modellen sparad som best_hgb_model.joblib


Utvärdera den bästa modellen från hyperparametoptimering 


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# -------------------------------------------------------
# Ladda modellen från joblib
# -------------------------------------------------------
hgb = joblib.load("best_hgb_model.joblib")
print("[JOBLIB] HGB-modellen laddad.")

# -------------------------------------------------------
# Prediktioner
# -------------------------------------------------------
preds = hgb.predict(X_test)
probs = hgb.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
acc = accuracy_score(Y_test, preds)
f1 = f1_score(Y_test, preds)
prec = precision_score(Y_test, preds)
rec = recall_score(Y_test, preds)
auc = roc_auc_score(Y_test, probs)
cm = confusion_matrix(Y_test, preds)

# -------------------------------------------------------
# Utskrifter
# -------------------------------------------------------
print("\n====================")
print("   HGB – METRICS")
print("====================")

print(f"Accuracy:      {acc:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"ROC-AUC:       {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(Y_test, preds))

[JOBLIB] HGB-modellen laddad.

   HGB – METRICS
Accuracy:      0.9714
F1-score:      0.8002
Precision:     0.9851
Recall:        0.6737
ROC-AUC:       0.9773

Confusion Matrix:
[[13712    13]
 [  416   859]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     13725
           1       0.99      0.67      0.80      1275

    accuracy                           0.97     15000
   macro avg       0.98      0.84      0.89     15000
weighted avg       0.97      0.97      0.97     15000



In [ ]:
# Ladda den sparade modellen
hgbc = joblib.load('hgbc_trained.pkl')

# Definiera scorer
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

# Beräkna cross-validation scores
cv_results = cross_validate(hgbc, X_train, Y_train, cv=5, scoring=scoring, n_jobs = -1)

# Skapa en lista med resultaten
results = []
for fold in range(5):
    results.append({
        'Fold': fold,
        'Accuracy': cv_results['test_accuracy'][fold],
        'Precision': cv_results['test_precision'][fold],
        'Recall': cv_results['test_recall'][fold],
        'F1-score': cv_results['test_f1'][fold]
    })

# Konvertera listan till en DataFrame
results = pd.DataFrame(results)

# Spara DataFrame med joblib
joblib.dump(results, 'cross_validation_results.pkl')

# Visa DataFrame
results